# Visual ML Foundations: Confusion Matrix & ROC-AUC Sweep
### Understanding True Positives, False Positives, and Threshold Invariance

---

## 1. Concept Overview
In competitive ML and clinical deployment, models output probabilities ($p \in [0, 1]$), not hard decisions.
* **Accuracy is Misleading**: On a 95% negative dataset, predicting all 0 gives 95% accuracy but saves 0 patients.
* **ROC Curve**: Plots **True Positive Rate (Sensitivity)** vs. **False Positive Rate (1 - Specificity)** across every possible decision threshold $\tau \in [0, 1]$.
* **ROC-AUC (Area Under Curve)**: Measures the probability that the model ranks a randomly chosen positive case higher than a randomly chosen negative case.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix

np.random.seed(42)
# True binary labels (50 positive, 450 negative)
y_true = np.concatenate([np.ones(50), np.zeros(450)])

# Realistic predicted probabilities with some overlap
y_scores = np.concatenate([
    np.random.beta(a=6, b=2, size=50),  # Positives shift high
    np.random.beta(a=2, b=7, size=450)  # Negatives shift low
])

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

# Evaluate confusion matrix at a chosen threshold (tau = 0.5)
tau = 0.5
y_pred = (y_scores >= tau).astype(int)
cm = confusion_matrix(y_true, y_pred)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve Plot
ax1.plot(fpr, tpr, color='crimson', lw=2.5, label=f'Model ROC (AUC = {roc_auc:.3f})')
ax1.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.500)')
ax1.set_title("Receiver Operating Characteristic (ROC)", fontsize=12, fontweight='bold')
ax1.set_xlabel("False Positive Rate (1 - Specificity)")
ax1.set_ylabel("True Positive Rate (Sensitivity / Recall)")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="lower right")

# Confusion Matrix Heatmap
im = ax2.imshow(cm, cmap='Blues', interpolation='nearest')
ax2.set_title(f"Confusion Matrix (Threshold τ = {tau})", fontsize=12, fontweight='bold')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(['Pred Negative (0)', 'Pred Positive (1)'], fontweight='bold')
ax2.set_yticklabels(['True Negative (0)', 'True Positive (1)'], fontweight='bold')

for i in range(2):
    for j in range(2):
        ax2.text(j, i, f"{cm[i, j]}", ha="center", va="center", color="white" if cm[i, j] > 100 else "black", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f"Calculated Macro ROC-AUC: {roc_auc:.4f}")
